# Lab 8.6 &mdash; Explicit Handoff (Capped)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Hand off between nodes with Command(goto=...)
- Run a real create_agent specialist between handoffs
- Cap runaway loops with recursion_limit (the multi-agent max_iterations)

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Failure modes & observability](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-08-06")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Agents also coordinate by **explicit handoffs**: a node finishes its part and passes control on. LangGraph
expresses this with **`Command`** &mdash; a node returns `Command(goto="tech", update={...})` to both **update
shared state** and **route** to the next node in one move. The danger is a **handoff loop** (A &rarr; B &rarr;
A forever, deck slide 19), so every graph runs under a **`recursion_limit`** &mdash; the multi-agent
`max_iterations`. Here each handoff node runs a **real `create_agent` specialist**, then hands off; a
deliberately looping pair shows the cap stopping a runaway team.

In [ ]:
# Each node returns Command(goto=..., update=...): state update + routing in one step.
print("handoff: billing (real) -> tech (real) -> END, all under a recursion_limit")

## Build it
Complete the final handoff: `tech` is the last specialist, so it hands off to **`END`**. Each node runs a real
specialist, then returns a `Command`.

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_core.tools import tool
from langchain.agents import create_agent

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

@tool
def lookup_invoice(order_id: str) -> str:
    """Look up the charges on an order by its id, e.g. '4471'. Use for billing / charge / refund questions."""
    charges = INVOICES.get(order_id.strip(), [])
    return str(charges) if charges else "no charges found for that order"

@tool
def known_issues(symptom: str) -> str:
    """Look up a known technical issue by symptom keyword, e.g. 'crash' or 'login'. Use for tech problems."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v["bug"] + ": " + v["fix"]
    return "no known issue matched"

class FlowState(TypedDict):
    message: str
    findings: Annotated[list, add]

def build_specialist(tools, role):
    return create_agent(llm, tools, system_prompt=f"You are the {role} specialist. Use ONLY your own tools; answer in one sentence.")

def handoff_node(role, tools, goto):
    """A specialist node that does its work, then HANDS OFF to the next node (or END)."""
    agent = build_specialist(tools, role)
    def node(state):
        r = agent.invoke({"messages": [("user", state["message"])]}, config={"recursion_limit": 8})
        return Command(goto=goto, update={"findings": [f"{role}: " + r["messages"][-1].content]})
    return node

# Workflow (Command handoffs -- routed at RUNTIME, not static edges):
#
#   START -> billing --Command(goto)--> tech --Command(goto)--> END
#   (a runaway ping <-> pong pair is stopped by recursion_limit)
def build_flow():
    g = StateGraph(FlowState)
    g.add_node("billing", handoff_node("billing", [lookup_invoice], goto="tech"))   # billing -> tech
    g.add_node("tech", handoff_node("tech", [known_issues], goto=___))          # TODO: tech is last -> hand off to END
    g.add_edge(START, "billing")
    return g.compile()

try:
    # The runaway guard is deterministic -- demo it offline (no model call).
    # A polite looping pair would hand back and forth forever; recursion_limit stops it.
    def ping(state) -> Command: return Command(goto="pong", update={"findings": ["ping"]})
    def pong(state) -> Command: return Command(goto="ping", update={"findings": ["pong"]})
    loop = StateGraph(FlowState)
    loop.add_node("ping", ping); loop.add_node("pong", pong); loop.add_edge(START, "ping")
    try:
        loop.compile().invoke({"message": "x", "findings": []}, config={"recursion_limit": 5})
    except Exception as e:
        print("looping pair stopped by recursion_limit ->", type(e).__name__)
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Build the flow and run one ticket: billing does its part and hands off to tech, which hands off to END. Read the findings each node contributed.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. Multi-agent runs make several model calls &mdash; keep live runs light on the free tier._

In [ ]:
if not groq_ready():
    print("GROQ_API_KEY not set -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        flow = build_flow()
        state = flow.invoke(
            {"message": "I was charged twice for 4471 and the app keeps crashing on login.", "findings": []},
            config={"recursion_limit": 8})
        print("handoff chain findings:")
        for f in state["findings"]:
            print(" -", f)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Each node returns `Command(goto=..., update=...)` &mdash; **state update and routing in one step**; `billing` hands to `tech`, `tech` hands to `END`.
- The looping `ping`/`pong` pair is stopped by **`recursion_limit`** (a `GraphRecursionError`) &mdash; the framework's built-in runaway guard, no manual counter.
- Lower the cap and it stops sooner: `recursion_limit` is your dial between *let the team finish* and *never run away*.

## Your turn (open task &mdash; no grader)
Make `billing` hand off **conditionally** &mdash; return `Command(goto="tech")` for a two-intent ticket but
`Command(goto=END)` for a billing-only one (branch on `state["message"]`). **What good looks like:** a normal ticket
terminates under the cap, while the `ping`/`pong` loop is *always* stopped by `recursion_limit` &mdash; no polite
pair of nodes hands back and forth forever.

---
### Key takeaway
Command(goto=...) updates state and routes in one step -- the framework's handoff -- and recursion_limit caps runaway loops without a manual counter. Explicit handoffs plus a cap keep a team from becoming a mob. Next: deciding when specialists disagree.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>